# KFV / FGO 实验

与本地共用 `core / model / data / config`。前四项运行 Python 算法，第五项只比较已有 MATLAB/Python 结果文件，全部不需要启动 MATLAB。

选择 github 获取已发布源码，或选择 upload 上传 GitHub 下载的源码 ZIP / 手工压缩的源码目录（排除环境、缓存和历史结果）；本地 Jupyter 可选择 existing。

In [ ]:
# @title 1. 加载代码（重复执行不会删除已有目录）
import os
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

SOURCE_MODE = "upload"  # @param ["upload", "existing", "github"]
WORKSPACE_PATH = ""  # @param {type:"string"}
REPO_URL = "https://github.com/Baoshan-Song/KFV-FGO-Comparison.git"  # @param {type:"string"}
BRANCH = "python_colab"  # @param {type:"string"}

if SOURCE_MODE == "existing":
    WORKSPACE = Path(WORKSPACE_PATH or Path.cwd()).expanduser().resolve()
    if (WORKSPACE / "kfv_fgo").is_dir():
        WORKSPACE = WORKSPACE / "kfv_fgo"
elif SOURCE_MODE == "github":
    checkout = Path(tempfile.mkdtemp(prefix="kfv-fgo-github-")) / "repository"
    subprocess.run(["git", "clone", "--branch", BRANCH, "--", REPO_URL, str(checkout)], check=True)
    WORKSPACE = checkout / "kfv_fgo"
elif SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
        raise ValueError("请仅上传一个源码 ZIP 文件")
    import io
    destination = Path(tempfile.mkdtemp(prefix="kfv-fgo-upload-"))
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if (not target.is_relative_to(destination) or "\\" in member.filename
                    or ":" in member.filename or (member.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError("ZIP contains an unsafe path")
        archive.extractall(destination)
    candidates = [p.parent for p in destination.rglob("pyproject.toml")
                  if (p.parent / "core/estimator.py").is_file()
                  and (p.parent / "examples/compare_matlab_results.py").is_file()]
    if len(candidates) != 1:
        raise ValueError("源码 ZIP 必须包含且仅包含一个完整的 kfv_fgo 工作目录")
    WORKSPACE = candidates[0]
else:
    raise ValueError("Unknown SOURCE_MODE")

if not (WORKSPACE / "core/estimator.py").is_file():
    raise FileNotFoundError("请选择重排后的 kfv_fgo 工作目录；上游旧分支尚不包含本次修改。")
print("工作目录:", WORKSPACE)


In [ ]:
# @title 2. 安装到当前 notebook 的 Python 环境
# 实测依赖固定 pyrtklib==0.2.7；验证环境为 Python 3.12。
INSTALL_REAL = True  # @param {type:"boolean"}
extras = "real" if INSTALL_REAL else "plots"
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{WORKSPACE}[{extras}]"], check=True)
# 使本单元刚安装的 editable package 可被当前内核正常导入。
import site
for directory in site.getsitepackages():
    site.addsitedir(directory)
# ZIP/checkout 的标准包目录使用普通搜索路径，避免内核保留旧 editable finder。
if WORKSPACE.name == "kfv_fgo":
    sys.path.insert(0, str(WORKSPACE.parent))
import kfv_fgo
if Path(kfv_fgo.__file__).resolve().parent != WORKSPACE:
    raise RuntimeError("内核已经导入其他位置的 kfv_fgo，请重启内核后重新运行。")
print("Python:", sys.version.split()[0], "Package:", kfv_fgo.__file__)


In [ ]:
# @title 3. 结果目录与显示
from datetime import datetime
from IPython.display import Image, display
OUTPUTS = []

def new_output(name):
    output = WORKSPACE / "results" / (name + "-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
    OUTPUTS.append(output)
    return output

def show_output(output):
    print("结果:", output)
    for image in sorted(output.glob("*.png")):
        display(Image(filename=str(image)))


In [ ]:
# @title 01：SWFGO 仿真
from kfv_fgo.examples.example_sw_fgo_simulation import run
window = 5  # @param {type:"slider", min:1, max:20, step:1}
iterations = 1  # @param {type:"slider", min:1, max:10, step:1}
kernel = "none"  # @param ["none", "huber", "cauchy", "tukey"]
output = new_output("01-simulation")
run(output=output, window=window, iterations=iterations, kernel=kernel)
show_output(output)


In [ ]:
# @title 02：SWFGO 实测
from kfv_fgo.examples.example_sw_fgo_real import run
window = 5  # @param {type:"slider", min:1, max:20, step:1}
iterations = 5  # @param {type:"slider", min:1, max:10, step:1}
kernel = "none"  # @param ["none", "huber", "cauchy", "tukey"]
DATA_DIR = ""  # @param {type:"string"}
if INSTALL_REAL:
    output = new_output("02-real")
    run(output=output, window=window, iterations=iterations, kernel=kernel, data_dir=DATA_DIR or None)
    show_output(output)
else:
    print("请开启 INSTALL_REAL 并执行安装单元后运行实测。")


In [ ]:
# @title 03：KFV / FGO 模板对比 仿真
from kfv_fgo.examples.example_kfv_fgo_simulation import run
mode = "all"  # @param ["all", "EKF", "iEKF", "rEKF", "riEKF"]
output = new_output("03-simulation")
run(output=output, modes=("EKF", "iEKF", "rEKF", "riEKF") if mode == "all" else (mode,))
show_output(output)


In [ ]:
# @title 04：KFV / FGO 模板对比 实测
from kfv_fgo.examples.example_kfv_fgo_real import run
mode = "all"  # @param ["all", "EKF", "iEKF", "rEKF", "riEKF"]
DATA_DIR = ""  # @param {type:"string"}
if INSTALL_REAL:
    output = new_output("04-real")
    run(output=output, modes=("EKF", "iEKF", "rEKF", "riEKF") if mode == "all" else (mode,), data_dir=DATA_DIR or None)
    show_output(output)
else:
    print("请开启 INSTALL_REAL 并执行安装单元后运行实测。")


## MATLAB 结果对比（仿真和实测共用）

只读取 `.mat` 和 `.npz`，不调用 MATLAB。默认使用 `tests/fixtures/matlab_simulation.mat`；将第三项实验输出的 `KFV_EKF.npz` 路径填入 PYTHON_FILE。实测改用 `tests/fixtures/matlab_real.mat` 和第四项的对应输出，也可在 Colab 左侧上传自己的 MATLAB 结果。旧文件可选 `result_ekf`，此前参考导出可选 `reference.runs.KFV_EKF`。多组结果必须指定对应字段；实测需要同一时间尺度的数值时间戳。

In [ ]:
# @title 05：比较已有 MATLAB/Python 结果
from kfv_fgo.examples.compare_matlab_results import run
COMPARE_RESULTS = False  # @param {type:"boolean"}
MATLAB_FILE = "tests/fixtures/matlab_simulation.mat"  # @param {type:"string"}
PYTHON_FILE = ""  # @param {type:"string"}
MATLAB_KEY = "reference.runs.KFV_EKF"  # @param {type:"string"}
MATLAB_TIME_KEY = ""  # @param {type:"string"}
DT_IF_MISSING = ""  # @param {type:"string"}
if COMPARE_RESULTS:
    output = new_output("05-matlab-results")
    if not PYTHON_FILE.strip():
        raise ValueError("请填写已生成的 Python NPZ 结果路径")
    matlab_path = WORKSPACE / MATLAB_FILE
    python_path = WORKSPACE / PYTHON_FILE
    report = run(matlab_path, python_path, output,
                 matlab_key=MATLAB_KEY or None, matlab_time_key=MATLAB_TIME_KEY or None,
                 dt=float(DT_IF_MISSING) if DT_IF_MISSING else None)
    show_output(output)
else:
    print("填写结果路径，开启 COMPARE_RESULTS 后执行；无需 MATLAB 环境。")


In [ ]:
# @title 下载本轮结果
DOWNLOAD_RESULTS = False  # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    archive_path = WORKSPACE / "results" / "notebook-results.zip"
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for output in OUTPUTS:
            for path in output.rglob("*"):
                if path.is_file():
                    archive.write(path, path.relative_to(WORKSPACE / "results").as_posix())
    try:
        from google.colab import files
    except ImportError:
        print(archive_path)
    else:
        files.download(str(archive_path))
